# Corrected jointly stigmatic lens — optical target and acoustic verification

The two surfaces now share one laboratory intermediate conjugate, following Silva-Lora & Torres (2020), section 3. This corrects the earlier independent-target prescription. **Optical construction, acoustic force fitting, coupled equilibrium and formation are different results.** No target projection below is presented as a formed lens.

Working prescription: indices 1.33 → 1.50 → 1.33; object/intermediate/image z = −101/+201/+151 mm; front/back vertices −1.15/+1.12 mm; both clear radii 2 mm. Materials remain hypothetical. Gravity is 9.81 m/s² in the mechanical calculations. The geometric spot gate is a maximum radius of 1 µm at the fixed detector, with every sampled ray transmitted.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
ROOT = Path.cwd()
if not (ROOT / 'src/acoustic_freeform').exists(): ROOT = ROOT.parent
STUDY = ROOT / 'artifacts/dual-stigmatic-2026-09-23'
AUDIT = STUDY / 'optical-audit'
validation = json.loads((AUDIT / 'validation.json').read_text())
print('Exact target, densest ray audit:')
print(json.dumps(validation['exact_target'][-1], indent=2))

## Both target surfaces and their representation error

These curves are the corrected desired surfaces and their spline approximation, **not a physical trajectory**. Surface errors use absolute laboratory height without removing piston or tilt. The target-pupil illumination cone is fixed and includes the axis and pupil rim. The plots have separately labelled horizontal and vertical scales.

In [ ]:
data = np.load(AUDIT / 'projection-52.npz')
r = data['radius_m']; pupil = r <= .002
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
for j, color in enumerate(['#1276bf', '#db6b16']):
    label = ['bottom/front', 'top/back'][j]
    axes[0].plot(r*1e3, ([-.001, .001][j] + data['target_heights_m'][j])*1e3, color=color, label=label)
    axes[1].plot(r[pupil]*1e3, (data['heights_m'][j,pupil]-data['target_heights_m'][j,pupil])*1e9, color=color, label=label)
axes[0].set(xlabel='Radius [mm]', ylabel='Laboratory z [mm]', title='Desired Cartesian pupils + fill annuli')
axes[0].axvline(2, color='gray', linestyle=':', label='clear-pupil edge')
axes[0].legend(fontsize=8)
axes[1].set(xlabel='Radius [mm]', ylabel='Representation error [nm]', title='Spline minus exact target; not formed error')
axes[1].legend(fontsize=8)
spot = data['spots_m']*1e6
axes[2].scatter(spot[:,0], spot[:,1], s=3)
axes[2].add_patch(plt.Circle((0,0), 1, fill=False, color='red', linestyle='--'))
axes[2].set(xlabel='Detector x [µm]', ylabel='Detector y [µm]', xlim=(-1.1,1.1), ylim=(-1.1,1.1), aspect='equal', title='Projected-target spot; red radius = 1 µm')
for ax in axes: ax.grid(alpha=.2)
plt.show()

In [ ]:
lines = ['| Surface elements | Front representation error [nm] | Back [nm] | Max geometric spot radius [µm] | All sampled rays transmit? |', '|---:|---:|---:|---:|---|']
for row in validation['spline_projection']:
    a, b = np.array(row['sampled_surface_max_error_m'])*1e9
    lines.append(f"| {row['surface_elements']} | {a:.6g} | {b:.6g} | {row['max_radius_m']*1e6:.6g} | {row['all_rays_transmitted']} |")
display(Markdown('\n'.join(lines)))

## Acoustic inverse attempts — not achieved surface errors

These are fixed-target, linearized compliance estimates from the acoustic traction residual. The command bounds are enforced before reporting these estimates. A termination flag is not proof of physical reachability or impossibility. The next section, when present, uses freshly solved coupled equilibria instead.

In [ ]:
lines = ['| Attempt | Frozen front error estimate [nm] | Back [nm] | Optimizer terminated successfully? |', '|---|---:|---:|---|']
for path in sorted(STUDY.glob('inverse-*/results.json')):
    row = json.loads(path.read_text())[-1]
    a,b = np.array(row['frozen_compliance_pupil_max_m'])*1e9
    lines.append(f"| {path.parent.name} | {a:.5g} | {b:.5g} | {row['optimizer_success']} |")
display(Markdown('\n'.join(lines)))
print('These are NOT forward surface errors or formation results.')

## Direct optical objective — optimization history, not physical time

The inverse now differentiates both ray intersections and both Snell refractions at the fixed detector. It minimizes cycle-mean pupil height and the **joint linearized spot**, with an additional annular force-fit penalty. Gravity remains in the mechanical load and compliance. The curves below are solver iterations, not lens formation. A surrogate pass must survive a fresh coupled wave/surface solve; acoustic feedback can invalidate the frozen compliance estimate.

In [ ]:
history_path = STUDY / 'conic-direct-448-7p2mhz-s104/history.json'
if history_path.exists():
    history = [x for x in json.loads(history_path.read_text()) if 'frozen_linearized_spot_radius_m' in x]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
    iterations = [x['iteration'] for x in history]
    heights = np.array([x['frozen_compliance_max_m'] for x in history])*1e9
    for j, color in enumerate(['#1276bf', '#db6b16']):
        axes[0].plot(iterations, heights[:,j], color=color, label=['front', 'back'][j])
    axes[0].axhline(10, color='red', linestyle='--', label='10 nm criterion')
    axes[0].set(xlabel='Optimization iteration; NOT physical time', ylabel='Frozen height estimate [nm]', yscale='log')
    axes[0].legend()
    axes[1].plot(iterations, [x['frozen_linearized_spot_radius_m']*1e6 for x in history])
    axes[1].axhline(1, color='red', linestyle='--')
    axes[1].set(xlabel='Optimization iteration; NOT physical time', ylabel='Linearized maximum spot radius [µm]', yscale='log')
    plt.show()
    print('Snapshot at notebook execution; forward success is NOT implied:')
    print(json.dumps(history[-1], indent=2))
else:
    print('No direct optical inverse history available.')

## Fresh coupled-equilibrium evidence

Commands are held fixed under refinement. A small force-balance residual alone is not a small error against the target. Neither stationary equilibrium nor a ray-tracing pass establishes transient formation or stability.

In [ ]:
found = False
for path in sorted(STUDY.glob('verify-*/results.json')):
    for case in json.loads(path.read_text()):
        for grid in case['grids']:
            found = True
            print(path.parent.name, grid['grid'])
            print('Actual stationary front/back maximum errors [nm]:', [row['max_error_m']*1e9 for row in grid['faces']])
            print('Fresh force-balance residual [m]:', grid['equilibrium']['compliance_residual_max_m'])
            print('Source work [W]:', grid['acoustics']['source_power_w'])
if not found:
    print('No completed coupled grid was available when this notebook was executed.')
for path in sorted(STUDY.glob('optics-verify-*/validation.json')):
    print(path.parent.name)
    print(path.read_text())

## Actual stationary lens and 448-region apparatus

This is the saved **forward equilibrium**, not the desired surface or an optimizer iterate. The two surfaces and their errors use the same saved state as the spot diagram. The apparatus comprises ideal full-azimuth source annuli and sidewall bands, not 448 fabricated point speakers. The 3D panel has equal physical scale; the error panel explicitly converts metres to nanometres. This stationary picture does not demonstrate formation or stability.

In [ ]:
from acoustic_freeform.dual.config import DualConfig
from acoustic_freeform.dual.surface import DualSurface, CartesianPatch
from acoustic_freeform.dual.sources import source_regions
from acoustic_freeform.dual.optics import trace_pair
campaign = STUDY / 'verify-direct-448'
if (campaign / 'results.json').exists():
    inputs = json.loads((campaign / 'config.json').read_text())
    record = json.loads((campaign / 'results.json').read_text())[0]
    grid = record['grids'][-1]
    cfg = DualConfig(**grid['numerics']); space = DualSurface(cfg)
    state = np.load(campaign / record['case'] / (grid['grid'] + '-state.npz'))
    q = state['coefficients_m']; drive = state['source_velocity_m_s']
    targets = [CartesianPatch(cfg,j,**f) for j,f in enumerate(inputs['cases'][0]['faces'])]
    r = np.linspace(0,cfg.radius_m,401); rp = np.linspace(0,cfg.clear_radius_m,2001)
    rays = np.linspace(0,cfg.clear_radius_m,4001)
    trace = trace_pair(space,q,[1.33,1.5,1.33],-.101,.151,launch_radius_m=rays,launch_height_m=cfg.levels_m[1]+targets[0].evaluate(rays))
    fig = plt.figure(figsize=(14,4.3),constrained_layout=True)
    ax = fig.add_subplot(131,projection='3d'); err = fig.add_subplot(132); spot = fig.add_subplot(133)
    theta = np.linspace(0,2*np.pi,65)
    for region in source_regions(cfg):
        radius = (region['r_min_m']+region['r_max_m'])/2
        z = (region['z_min_m']+region['z_max_m'])/2
        ax.plot(radius*np.cos(theta)*1e3,radius*np.sin(theta)*1e3,np.full_like(theta,z)*1e3,color=plt.cm.viridis(abs(drive[region['channel']])),alpha=.16,linewidth=.4)
    rr,tt = np.meshgrid(np.linspace(0,cfg.radius_m,41),theta)
    for j,color in enumerate(['#1276bf','#db6b16']):
        ax.plot_surface(rr*np.cos(tt)*1e3,rr*np.sin(tt)*1e3,(cfg.levels_m[j+1]+space.evaluate(q[j],rr))*1e3,color=color,alpha=.75,linewidth=0)
        err.plot(rp*1e3,(space.evaluate(q[j],rp)-targets[j].evaluate(rp))*1e9,color=color,label=['front/bottom','back/top'][j])
    ax.set(xlabel='x [mm]',ylabel='y [mm]',zlabel='z [mm]',title='Saved equilibrium; equal physical scale',xlim=(-4,4),ylim=(-4,4),zlim=(-3,3))
    ax.set_box_aspect((8,8,6))
    err.axhspan(-10,10,color='green',alpha=.2,label='±10 nm band')
    err.set(xlabel='Pupil radius [mm]',ylabel='Actual height minus target [nm]',title='Both faces: absolute height error'); err.legend(fontsize=8)
    xy = trace['spots_m']*1e6
    spot.scatter(xy[:,0],xy[:,1],s=2,alpha=.6)
    spot.add_patch(plt.Circle((0,0),1,fill=False,color='red'))
    spot.set(xlabel='Detector x [µm]',ylabel='Detector y [µm]',aspect='equal',title='Full two-interface geometric spot')
    plt.show()
    print('Grid:',grid['grid'],'; all rays transmitted:',trace['all_rays_transmitted'])
    print('Maximum geometric radius [µm]:',trace['max_radius_m']*1e6)
    print('No piston removal, recentering, refocusing or discarded-ray pass.')
else:
    print('No saved direct-objective equilibrium is available.')

## Independent numerical checks

These checks hold the physical command fixed. Quadrature and wave-mesh changes below are converted into mechanical-compliance load diagnostics at the same geometry; they are **not newly solved shapes**. Cache agreement compares a fresh single-command wave solve with the independently assembled all-source quadratic operator. None of these checks certifies stability.

In [ ]:
for name in ['fixed-quadrature-224', 'fixed-wave-224', 'kernel-audit-direct-448']:
    path = STUDY / name / 'validation.json'
    if path.exists():
        print(name)
        rows = json.loads(path.read_text())
        if isinstance(rows, list):
            for row in rows:
                print(row['name'], 'pupil load-change proxy [nm]:', np.array(row['frozen_compliance_force_change_pupil_max_m'])*1e9)
        else:
            print(json.dumps(rows, indent=2))

## Formation animation status

The [previous executed two-face notebook](../artifacts/notebooks/06_two_face_formation.html) contains its physical-time animation embedded in the notebook. It remains an unsuccessful historical inviscid run for the old independent targets. It is **not** relabelled as the corrected lens forming. No successful corrected-target formation animation exists yet.

See [the correction and verification notes](../docs/joint-stigmatic-correction.md). Full configuration, source snapshots and numerical reports are in `artifacts/dual-stigmatic-2026-09-23/`. This study does not compute diffraction, window optics, real-material uncertainty, streaming, thermal feedback or experimental validation.